Name: Boga Petruska and Bence Szabo\
Class: ECBS5171 - Data Analysis 3\
Assignment: Assignment 2 - Fast Growth Prediction\
Date: Feb 15, 2026

Modeling Notebook

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import unicodedata
import re
import time
import warnings
import ast
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.model_selection import GridSearchCV, RepeatedKFold
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.tree import DecisionTreeRegressor
import xgboost as xgb
from xgboost import XGBRegressor
from collections import Counter
import shap

# Set display options
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 10)

In [2]:
csv_path = "../data/prepped/bisnode_firms_prepped.csv"
data = pd.read_csv(csv_path)

# data.to_csv("../data/prepped/bisnode_firms_prepped.csv", index=False)

In [4]:
# interaction terms
interactions = [
    ("ind2_cat", "age"), ("ind2_cat", "age2"),
    ("ind2_cat", "sales_mil_log"), ("ind2_cat", "ceo_age"),
    ("ind2_cat", "foreign_management"),
    ("sales_mil_log", "age"), ("sales_mil_log", "profit_loss_year_pl"),
    ("sales_mil_log", "foreign_management"),
]

for var1, var2 in interactions:
    col_name = f"{var1}*{var2}"
    data[col_name] = data[var1] * data[var2]

print(f"Interaction terms created: {len(interactions)}")

Interaction terms created: 8


In [5]:
# organize variable lists for modeling
rawvars = ["curr_assets", "curr_liab", "extra_exp", "extra_inc", 
           "extra_profit_loss", "fixed_assets", "inc_bef_tax", 
           "intang_assets", "inventories", "liq_assets", "material_exp",
           "personnel_exp", "profit_loss_year", "sales", "share_eq", 
           "subscribed_cap"]

qualityvars = ["balsheet_flag", "balsheet_length", "balsheet_notfullyear"]

engvar = ["total_assets_bs", "fixed_assets_bs", "liq_assets_bs", 
          "curr_assets_bs", "share_eq_bs", "subscribed_cap_bs",
          "intang_assets_bs", "extra_exp_pl", "extra_inc_pl",
          "extra_profit_loss_pl", "inc_bef_tax_pl", "inventories_pl",
          "material_exp_pl", "profit_loss_year_pl", "personnel_exp_pl"]

engvar2 = ["extra_profit_loss_pl_quad", "inc_bef_tax_pl_quad",
           "profit_loss_year_pl_quad", "share_eq_bs_quad"]

# Collect all flag variables dynamically
engvar3 = [col for col in data.columns 
           if col.endswith(("flag_low", "flag_high", "flag_zero"))]

hr = ["female", "ceo_age", "flag_high_ceo_age", "flag_low_ceo_age",
      "flag_miss_ceo_age", "ceo_count", "foreign_management"]

firm = ["age", "age2", "new", "ind2_cat", "manufacturing_flg", 
        "services_flg", "urban_m"]

interactions_list = [f"{v1}*{v2}" for v1, v2 in interactions]

print(f"Variable counts:")
print(f"  Raw: {len(rawvars)}, Quality: {len(qualityvars)}")
print(f"  Engineered: {len(engvar)}, Squared: {len(engvar2)}")
print(f"  Flags: {len(engvar3)}, HR: {len(hr)}, Firm: {len(firm)}")
print(f"  Interactions: {len(interactions_list)}")
print(f"  Total features: {len(rawvars + qualityvars + engvar + engvar2 + engvar3 + hr + firm + interactions_list)}")

Variable counts:
  Raw: 16, Quality: 3
  Engineered: 15, Squared: 4
  Flags: 12, HR: 7, Firm: 7
  Interactions: 8
  Total features: 72
